In [1]:
import duckdb
from functions.evaluation import evaluate
from networks.cnn_network import CNNModel
from networks.cnn_network import build_dataloaders


con = duckdb.connect('../capillary.db')
df = con.execute(""" 
                 SELECT row_id, value,fractions, boundaries,albumin,antitrypsin,orosomukoid,haptoglobin,crp,igg,iga,igm, label, set, interpretation
                 FROM protein_data
                 WHERE value IS NOT NULL
                 AND observation_nr = 1
                 AND analysis IS NOT NULL
                 AND protein_value IS NOT NULL
                 """).df()
con.close()


train_rows = df[df['set'] == 'train']
val_rows   = df[df['set'] == 'val']
test_rows  = df[df['set'] == 'test']

drop_indices = train_rows[train_rows['label'] == 0].sample(frac=0.7).index

train_rows = train_rows.drop(drop_indices)
train_rows = train_rows[train_rows['label'].isin([0,1])]
val_rows = val_rows[val_rows['label'].isin([0,1])]

print(f"Antal utan m-komponent i träningsdatan: {len(train_rows[train_rows['label'] == 0])}")
print(f"Antal med m-komponent i träningsdatan: {len(train_rows[train_rows['label'] == 1])}")

CNN = CNNModel()
cnn_train_dl, cnn_val_dl, _ = build_dataloaders(train_rows, val_rows, val_rows)
#CNN.reset_weights()
#CNN.retrain(cnn_train_dl,cnn_val_dl,patience=15)

Antal utan m-komponent i träningsdatan: 17903
Antal med m-komponent i träningsdatan: 2553
Total parameters: 82,146


In [2]:
con = duckdb.connect('../capillary.db')
df = con.execute(""" 
                 SELECT row_id, value,fractions, boundaries,albumin,antitrypsin,orosomukoid,haptoglobin,crp,igg,iga,igm, label, set, interpretation
                 FROM protein_data
                 WHERE value IS NOT NULL
                 AND observation_nr = 1
                 AND analysis IS NOT NULL
                 AND protein_value IS NOT NULL
                 """).df()
con.close()


k_fold_rows = df[df['set'].isin(['train','val']) ].copy()
k_fold_rows = k_fold_rows[k_fold_rows['label'].isin([0,1])]
drop_indices = k_fold_rows[k_fold_rows['label'] == 0].sample(frac=0.7).index
k_fold_rows = k_fold_rows.drop(drop_indices)
print(f"Totalt antal utan m-komponent i K-fold poolen: {len(k_fold_rows[k_fold_rows['label'] == 0])}")
print(f"Totalt antal med m-komponent i K-fold poolen: {len(k_fold_rows[k_fold_rows['label'] == 1])}")

test_rows  = df[df['set'] == 'test']


CNN = CNNModel()
CNN.retrain_with_k_fold(k_fold_rows)

Totalt antal utan m-komponent i K-fold poolen: 18848
Totalt antal med m-komponent i K-fold poolen: 2692
--- Startar 10-Fold Cross Validation ---

 FOLD 1/10
  -> ny bästa modell sparad till ../models/convolution_model_fold1.pth
Epoch   0 | train: 1.1886 | val: 0.6631 | acc: 35.16% | AUC: 0.683  | LR: 0.001
  -> ny bästa modell sparad till ../models/convolution_model_fold1.pth
Epoch   1 | train: 0.5699 | val: 0.5199 | acc: 89.60% | AUC: 0.825  | LR: 0.001
  -> ny bästa modell sparad till ../models/convolution_model_fold1.pth
Epoch   2 | train: 0.4679 | val: 0.4567 | acc: 89.18% | AUC: 0.861  | LR: 0.001
  -> ny bästa modell sparad till ../models/convolution_model_fold1.pth
Epoch   3 | train: 0.4361 | val: 0.4242 | acc: 83.93% | AUC: 0.883  | LR: 0.001
  -> ny bästa modell sparad till ../models/convolution_model_fold1.pth
Epoch   4 | train: 0.4074 | val: 0.3861 | acc: 89.64% | AUC: 0.902  | LR: 0.001
  -> ny bästa modell sparad till ../models/convolution_model_fold1.pth
Epoch   5 | train

,fold,train_loss,val_loss,val_accuracy,val_auc,val_spec,val_sens,tn,fp,fn,tp
0,1,0.081427,0.178436,96.748723,0.985913,0.977707,0.895911,1842,42,28,241
1,2,0.157277,0.168936,93.779016,0.980953,0.936340,0.947955,1765,120,14,255
2,3,0.087756,0.106101,96.745700,0.988897,0.970776,0.944238,1827,55,15,254
3,4,0.094756,0.130574,95.311049,0.987904,0.953316,0.951673,1797,88,13,256
4,5,0.112765,0.156449,95.914578,0.981722,0.965517,0.914498,1820,65,23,246
5,6,0.289042,0.312196,89.925720,0.925161,0.911936,0.810409,1719,166,51,218
6,7,0.089261,0.191888,96.239554,0.976238,0.972414,0.892193,1833,52,29,240
7,8,0.084231,0.181748,95.264624,0.985631,0.957560,0.918216,1805,80,22,247
8,9,0.086606,0.158239,94.194148,0.985130,0.940520,0.951852,1771,112,13,257
9,10,0.096105,0.105676,96.237808,0.992453,0.966012,0.937037,1819,64,17,253


In [3]:
test_rows = test_rows[test_rows['label'].isin([0,1])]
result = CNN.predict(test_rows)
_ = evaluate(result)

KeyError: 'prediction'